## Read Bronze tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
spark.sql(f"create schema if not exists {SILVER}")

encounters_bronze = spark.table(f"{BRONZE}.bronze_patient_encounters")
labs_bronze = spark.table(f"{BRONZE}.bronze_lab_test_events")
transfers_bronze = spark.table(f"{BRONZE}.bronze_bed_transfer_events")

patients_bronze = spark.table(f"{BRONZE}.bronze_patients")
departments_bronze = spark.table(f"{BRONZE}.bronze_departments")
beds_bronze = spark.table(f"{BRONZE}.bronze_beds")

print("Encounters:", encounters_bronze.count())
print("Lab events:", labs_bronze.count())
print("Bed transfers:", transfers_bronze.count())
print("Patients:", patients_bronze.count())
print("Departments:", departments_bronze.count())
print("Beds:", beds_bronze.count())

Encounters: 12376
Lab events: 758
Bed transfers: 392
Patients: 183
Departments: 192
Beds: 201


## Clean master tables

In [0]:
patients_silver = (
    patients_bronze
    .select(
        F.trim("patient_id").alias("patient_id"),
        F.trim("synthetic_patient_code").alias("synthetic_patient_code"),
        F.trim("age_band").alias("age_band"),
        F.upper(F.trim("sex")).alias("sex"),
        F.trim("city").alias("city"),
        F.to_date("registration_date").alias("registration_date"),
        F.upper(F.trim("consent_flag")).alias("consent_flag"),
        F.upper(F.trim("insurance_type")).alias("insurance_type"),
        F.upper(F.trim("patient_status")).alias("patient_status"),
        F.trim("primary_language").alias("primary_language")
    )
)

departments_silver = (
    departments_bronze
    .select(
        F.trim("department_id").alias("department_id"),
        F.trim("department_name").alias("department_name"),
        F.trim("specialty").alias("specialty"),
        F.trim("hospital_site").alias("hospital_site"),
        F.col("floor").cast("int").alias("floor"),
        F.upper(F.trim("service_level")).alias("service_level"),
        F.upper(F.trim("active_flag")).alias("active_flag"),
        F.trim("head_clinician_id").alias("head_clinician_id"),
        F.col("capacity_beds").cast("int").alias("capacity_beds"),
        F.trim("region").alias("region")
    )
)

beds_silver = (
    beds_bronze
    .select(
        F.trim("bed_id").alias("bed_id"),
        F.trim("department_id").alias("department_id"),
        F.trim("ward").alias("ward"),
        F.upper(F.trim("bed_type")).alias("bed_type"),
        F.upper(F.trim("isolation_capable")).alias("isolation_capable"),
        F.upper(F.trim("bed_status")).alias("bed_status"),
        F.col("floor").cast("int").alias("floor"),
        F.trim("hospital_site").alias("hospital_site"),
        F.upper(F.trim("active_flag")).alias("active_flag"),
        F.to_date("effective_date").alias("effective_date")
    )
)

patients_silver = patients_silver.dropDuplicates(["patient_id"])
departments_silver = departments_silver.dropDuplicates(["department_id"])
beds_silver = beds_silver.dropDuplicates(["bed_id"])

print("Patients Silver:", patients_silver.count())
print("Departments Silver:", departments_silver.count())
print("Beds Silver:", beds_silver.count())

Patients Silver: 183
Departments Silver: 192
Beds Silver: 201


## Clean patient encounters

In [0]:
encounters_checked = (
    encounters_bronze
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("bed_id", F.trim("bed_id"))
    .withColumn("admission_ts_check", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts_check", F.to_timestamp("discharge_ts"))
    .withColumn("encounter_type", F.upper(F.trim("encounter_type")))
    .withColumn("triage_priority", F.upper(F.trim("triage_priority")))
    .withColumn("encounter_status", F.upper(F.trim("encounter_status")))
    .withColumn("attending_clinician_id", F.trim("attending_clinician_id"))
    .withColumn("admission_source", F.upper(F.trim("admission_source")))
    .withColumn("diagnosis_group", F.upper(F.trim("diagnosis_group")))
)

encounters_checked = encounters_checked.withColumn(
    "dq_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "MISSING_PATIENT_ID"),
        F.when(F.col("department_id").isNull() | (F.col("department_id") == ""), "MISSING_DEPARTMENT_ID"),
        F.when(F.col("admission_ts_check").isNull(), "INVALID_ADMISSION_TIMESTAMP")
    )
)

encounters_rejected = encounters_checked.filter(F.col("dq_reason") != "")
encounters_valid = encounters_checked.filter(F.col("dq_reason") == "")

patient_keys = patients_silver.select("patient_id").distinct()
department_keys = departments_silver.select("department_id").distinct()
bed_keys = beds_silver.select("bed_id").distinct()

encounters_valid = (
    encounters_valid
    .join(patient_keys.withColumn("patient_exists", F.lit(True)), "patient_id", "left")
    .join(department_keys.withColumn("department_exists", F.lit(True)), "department_id", "left")
    .join(bed_keys.withColumn("bed_exists", F.lit(True)), "bed_id", "left")
)

encounters_valid = encounters_valid.withColumn(
    "dq_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("patient_exists").isNull(), "UNKNOWN_PATIENT_ID"),
        F.when(F.col("department_exists").isNull(), "UNKNOWN_DEPARTMENT_ID"),
        F.when(F.col("bed_id").isNotNull() & F.col("bed_exists").isNull(), "UNKNOWN_BED_ID"),
        F.when(
            F.col("discharge_ts_check").isNotNull() &
            (F.col("discharge_ts_check") < F.col("admission_ts_check")),
            "DISCHARGE_BEFORE_ADMISSION"
        )
    )
)

encounters_rejected = encounters_rejected.unionByName(
    encounters_valid.filter(F.col("dq_reason") != ""),
    allowMissingColumns=True
)

encounters_valid = encounters_valid.filter(F.col("dq_reason") == "")

encounter_window = (
    Window.partitionBy("encounter_id")
    .orderBy(F.col("ingestion_timestamp").desc(), F.col("source_file").asc())
)

encounters_ranked = encounters_valid.withColumn(
    "row_num", F.row_number().over(encounter_window)
)

encounters_duplicate_audit = encounters_ranked.filter(F.col("row_num") > 1)
encounters_silver = encounters_ranked.filter(F.col("row_num") == 1)

encounters_silver = encounters_silver.select(
    "encounter_id", "patient_id", "department_id", "bed_id",
    F.col("admission_ts_check").alias("admission_ts"),
    F.col("discharge_ts_check").alias("discharge_ts"),
    "encounter_type", "triage_priority", "encounter_status",
    "attending_clinician_id", "admission_source", "diagnosis_group",
    "source_file", "ingestion_timestamp"
)

print("Encounters Silver:", encounters_silver.count())
print("Encounters rejected:", encounters_rejected.count())
print("Encounter duplicates:", encounters_duplicate_audit.count())

Encounters Silver: 12329
Encounters rejected: 0
Encounter duplicates: 47


## Clean lab test events

In [0]:
labs_checked = (
    labs_bronze
    .withColumn("lab_event_id", F.trim("lab_event_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("test_code", F.upper(F.trim("test_code")))
    .withColumn("ordered_ts_check", F.to_timestamp("ordered_ts"))
    .withColumn("collected_ts_check", F.to_timestamp("collected_ts"))
    .withColumn("result_ts_check", F.to_timestamp("result_ts"))
    .withColumn("result_status", F.upper(F.trim("result_status")))
    .withColumn("priority", F.upper(F.trim("priority")))
)

labs_checked = labs_checked.withColumn(
    "dq_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("lab_event_id").isNull() | (F.col("lab_event_id") == ""), "MISSING_LAB_EVENT_ID"),
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("ordered_ts_check").isNull(), "INVALID_ORDERED_TIMESTAMP")
    )
)

labs_rejected = labs_checked.filter(F.col("dq_reason") != "")
labs_valid = labs_checked.filter(F.col("dq_reason") == "")

encounter_keys = encounters_silver.select("encounter_id").distinct()
labs_valid = labs_valid.join(
    encounter_keys.withColumn("encounter_exists", F.lit(True)),
    "encounter_id", "left"
)

labs_valid = labs_valid.withColumn(
    "dq_reason",
    F.when(F.col("encounter_exists").isNull(), "UNKNOWN_ENCOUNTER_ID").otherwise("")
)

labs_rejected = labs_rejected.unionByName(
    labs_valid.filter(F.col("dq_reason") != ""),
    allowMissingColumns=True
)
labs_valid = labs_valid.filter(F.col("dq_reason") == "")

lab_window = Window.partitionBy("lab_event_id").orderBy(
    F.col("ingestion_timestamp").desc(), F.col("source_file").asc()
)
labs_ranked = labs_valid.withColumn("row_num", F.row_number().over(lab_window))
labs_duplicate_audit = labs_ranked.filter(F.col("row_num") > 1)

labs_silver = (
    labs_ranked.filter(F.col("row_num") == 1)
    .select(
        "lab_event_id", "encounter_id", "patient_id", "test_code",
        F.col("ordered_ts_check").alias("ordered_ts"),
        F.col("collected_ts_check").alias("collected_ts"),
        F.col("result_ts_check").alias("result_ts"),
        "result_status", "department_id", "priority",
        "source_file", "ingestion_timestamp"
    )
)

print("Lab Silver:", labs_silver.count())
print("Lab rejected:", labs_rejected.count())
print("Lab duplicates:", labs_duplicate_audit.count())

Lab Silver: 758
Lab rejected: 0
Lab duplicates: 0


## Clean bed transfer events

In [0]:
transfers_checked = (
    transfers_bronze
    .withColumn("transfer_id", F.trim("transfer_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("from_bed_id", F.trim("from_bed_id"))
    .withColumn("to_bed_id", F.trim("to_bed_id"))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("transfer_ts_check", F.to_timestamp("transfer_ts"))
    .withColumn("recorded_ts_check", F.to_timestamp("recorded_ts"))
    .withColumn("transfer_reason", F.upper(F.trim("transfer_reason")))
    .withColumn("transfer_status", F.upper(F.trim("transfer_status")))
)

transfers_checked = transfers_checked.withColumn(
    "dq_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("transfer_id").isNull() | (F.col("transfer_id") == ""), "MISSING_TRANSFER_ID"),
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("transfer_ts_check").isNull(), "INVALID_TRANSFER_TIMESTAMP")
    )
)

transfers_rejected = transfers_checked.filter(F.col("dq_reason") != "")
transfers_valid = transfers_checked.filter(F.col("dq_reason") == "")

transfers_valid = transfers_valid.join(
    encounter_keys.withColumn("encounter_exists", F.lit(True)),
    "encounter_id", "left"
)

transfers_valid = transfers_valid.withColumn(
    "dq_reason",
    F.when(F.col("encounter_exists").isNull(), "UNKNOWN_ENCOUNTER_ID").otherwise("")
)

transfers_rejected = transfers_rejected.unionByName(
    transfers_valid.filter(F.col("dq_reason") != ""),
    allowMissingColumns=True
)
transfers_valid = transfers_valid.filter(F.col("dq_reason") == "")

transfer_window = Window.partitionBy("transfer_id").orderBy(
    F.col("ingestion_timestamp").desc(), F.col("source_file").asc()
)
transfers_ranked = transfers_valid.withColumn(
    "row_num", F.row_number().over(transfer_window)
)
transfers_duplicate_audit = transfers_ranked.filter(F.col("row_num") > 1)

transfers_silver = (
    transfers_ranked.filter(F.col("row_num") == 1)
    .select(
        "transfer_id", "encounter_id", "patient_id",
        "from_bed_id", "to_bed_id",
        F.col("transfer_ts_check").alias("transfer_ts"),
        "transfer_reason", "department_id", "transfer_status",
        F.col("recorded_ts_check").alias("recorded_ts"),
        "source_file", "ingestion_timestamp"
    )
)

print("Transfers Silver:", transfers_silver.count())
print("Transfers rejected:", transfers_rejected.count())
print("Transfer duplicates:", transfers_duplicate_audit.count())

Transfers Silver: 392
Transfers rejected: 0
Transfer duplicates: 0


## Save Silver tables

In [0]:
silver_tables = {
    "silver_patients": patients_silver,
    "silver_departments": departments_silver,
    "silver_beds": beds_silver,
    "silver_patient_encounters": encounters_silver,
    "silver_lab_test_events": labs_silver,
    "silver_bed_transfer_events": transfers_silver
}

for table_name, dataframe in silver_tables.items():
    (dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{table_name}")
    )

rejected_records = (
    encounters_rejected.withColumn("source_entity", F.lit("PATIENT_ENCOUNTERS"))
    .unionByName(
        labs_rejected.withColumn("source_entity", F.lit("LAB_TEST_EVENTS")),
        allowMissingColumns=True
    )
    .unionByName(
        transfers_rejected.withColumn("source_entity", F.lit("BED_TRANSFER_EVENTS")),
        allowMissingColumns=True
    )
)

duplicate_audit = (
    encounters_duplicate_audit.withColumn("source_entity", F.lit("PATIENT_ENCOUNTERS"))
    .unionByName(
        labs_duplicate_audit.withColumn("source_entity", F.lit("LAB_TEST_EVENTS")),
        allowMissingColumns=True
    )
    .unionByName(
        transfers_duplicate_audit.withColumn("source_entity", F.lit("BED_TRANSFER_EVENTS")),
        allowMissingColumns=True
    )
)

(rejected_records.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.silver_rejected_records")
)

(rejected_records.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.silver_quarantine")
)

(duplicate_audit.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.silver_duplicate_audit")
)

print("Silver tables saved successfully.")

Silver tables saved successfully.


## Validate Silver tables

In [0]:
for table_name in [
    "silver_patients",
    "silver_departments",
    "silver_beds",
    "silver_patient_encounters",
    "silver_lab_test_events",
    "silver_bed_transfer_events",
    "silver_rejected_records",
    "silver_quarantine",
    "silver_duplicate_audit"
]:
    dataframe = spark.table(f"{SILVER}.{table_name}")
    print(table_name, "| Records:", dataframe.count())

print(
    "Duplicate encounter IDs remaining:",
    spark.table(f"{SILVER}.silver_patient_encounters")
    .groupBy("encounter_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate lab event IDs remaining:",
    spark.table(f"{SILVER}.silver_lab_test_events")
    .groupBy("lab_event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate transfer IDs remaining:",
    spark.table(f"{SILVER}.silver_bed_transfer_events")
    .groupBy("transfer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_patients | Records: 183
silver_departments | Records: 192
silver_beds | Records: 201
silver_patient_encounters | Records: 12329
silver_lab_test_events | Records: 758
silver_bed_transfer_events | Records: 392
silver_rejected_records | Records: 0
silver_quarantine | Records: 0
silver_duplicate_audit | Records: 47
Duplicate encounter IDs remaining: 0
Duplicate lab event IDs remaining: 0
Duplicate transfer IDs remaining: 0
